# From a power spectrum to a model probability

This notebook follows one detection from beginning to end. The aim is to make the statistical calculation visible rather than treating `Detector` as a black box.

The steps are:

1. construct a cloud of plausible stellar properties;
2. generate stochastic power spectra whose true contents are known;
3. bin each spectrum on roughly the large-separation scale;
4. compare the data with three complete spectral models;
5. average each likelihood over the uncertain model parameters; and
6. turn the resulting evidences into posterior model probabilities.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from asterodetect import (
    AsteroScaleSamples,
    AstrophysicalInjectionFactory,
    Detector,
    GaussianEnvelope,
    HarveyComponent,
    NuisancePrior,
    ObservationModel,
    SpectralModel,
)
from asterodetect.asteroscale import ASTERO_SCALE_PARAMETERS

plt.style.use('seaborn-v0_8-whitegrid')

## 1. Plausible stars from AsteroScale

AsteroScale does not provide one exact star. It provides joint samples: each row is one internally consistent set of stellar and seismic quantities. Keeping the rows intact is important because quantities such as $\nu_{\max}$, $\Delta\nu$, amplitude, and granulation are correlated.

For this self-contained example we make an artificial subgiant-like sample cloud. A real analysis would use `AsteroScaleSamples.infer(...)`.

In [ ]:
rng = np.random.default_rng(7)
draws = 512
stellar_latent = rng.normal(size=draws)
amplitude_latent = -0.45 * stellar_latent + np.sqrt(1 - 0.45**2) * rng.normal(size=draws)

values = {name: np.ones(draws) for name in ASTERO_SCALE_PARAMETERS}
values.update(
    numax=800.0 * np.exp(0.04 * stellar_latent),
    dnu=50.0 * np.exp(0.025 * stellar_latent),
    FWHM_env=350.0 * np.exp(0.10 * stellar_latent),
    A_env=5.0 * np.exp(0.16 * amplitude_latent),
    A_gran=150.0 * np.exp(-0.12 * stellar_latent),
    b_gran_low=190.0 * np.exp(0.04 * stellar_latent),
    b_gran_high=720.0 * np.exp(0.04 * stellar_latent),
)
samples = AsteroScaleSamples(values)

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(samples.values['numax'], samples.values['dnu'], s=10, alpha=0.35)
ax.set(xlabel=r'$\nu_{\max}$ [$\mu$Hz]', ylabel=r'$\Delta\nu$ [$\mu$Hz]', title='Aligned AsteroScale-like rows');

## 2. Generate spectra with known contents

A power spectrum is stochastic: even when its mean spectrum is fixed, every realization looks different. The injection factory first builds a mean spectrum and then draws the noisy periodogram.

We generate two examples:

- **granulation:** white noise plus two broad Harvey-like granulation components;
- **oscillation:** the same background plus a comb of damped oscillation modes.

The detector is not told which case is which.

In [ ]:
factory = AstrophysicalInjectionFactory(
    samples,
    duration_days=27.4,
    cadence_seconds=120.0,
    white_noise=0.2,
)
cases = {
    truth: factory(
        truth,
        {'truth': truth, 'amplitude_scale': 1.0, 'window_profile': 'continuous'},
        np.random.default_rng(seed),
    )
    for truth, seed in [('granulation', 20), ('oscillation', 21)]
}
[(name, case.spectrum.frequency.size) for name, case in cases.items()]

## 3. Run the detector

The bin edges are chosen once from the AsteroScale prediction and then frozen. Adaptive importance sampling changes how the parameter integral is estimated; it does not alter the data or break apart the AsteroScale rows.

In [ ]:
observation = ObservationModel(integration_time_seconds=120.0)
detector = Detector(
    draws=128,
    pilot_draws=64,
    estimator='adaptive',
    nuisance_prior=NuisancePrior(),
    observation=observation,
)
results = {
    name: detector.run(case.spectrum, samples, rng=seed)
    for (name, case), seed in zip(cases.items(), [120, 121])
}
{name: result.probabilities for name, result in results.items()}

## 4. Compare the data with the three complete models

For a readable plot, the curves below use the median AsteroScale row and the known injected white-noise level. The detector itself is more careful: it integrates over many stellar rows and nuisance-parameter values.

Each coloured curve describes the **whole** spectrum. The models are not labels assigned independently to individual frequency bins.

In [ ]:
granulation_parameters = samples.granulation_parameters(observation)
harvey_components = tuple(
    HarveyComponent.from_rms_amplitude(
        np.median(granulation_parameters['amplitudes'][:, component]),
        np.median(granulation_parameters['frequencies'][:, component]),
    )
    for component in range(2)
)
envelope_parameters = samples.envelope_parameters(observation)
envelope = GaussianEnvelope(
    integrated_power=np.median(envelope_parameters['integrated_power']),
    numax=np.median(envelope_parameters['numax']),
    sigma=np.median(envelope_parameters['sigma']),
)
models = {
    'noise': SpectralModel(0.2),
    'granulation': SpectralModel(0.2, harvey_components),
    'oscillation': SpectralModel(0.2, harvey_components, envelope),
}

fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True, sharey=True)
colours = {'noise': '0.35', 'granulation': 'tab:orange', 'oscillation': 'tab:blue'}
for ax, (truth, result) in zip(axes, results.items()):
    spectrum = result.binned_spectrum
    ax.scatter(spectrum.frequency, spectrum.power, s=18, color='black', alpha=0.65, label='binned spectrum')
    for label, model in models.items():
        prediction = model.mean_binned_spectrum(spectrum.bin_lower, spectrum.bin_upper)
        ax.plot(spectrum.frequency, prediction, lw=2, color=colours[label], label=label)
    ax.set_yscale('log')
    ax.set_xlim(0, 1600)
    ax.set_ylabel('Power density')
    ax.set_title(f'Injected class: {truth}')
axes[0].legend(ncol=2)
axes[-1].set_xlabel(r'Frequency [$\mu$Hz]')
plt.tight_layout();

## 5. From likelihood to evidence to probability

For one parameter row $\theta_s$, Skuld calculates the likelihood of the complete binned spectrum. It then averages over plausible rows:

$$
p(D\mid M_k) \approx \frac{1}{N}\sum_{s=1}^{N} p(D\mid \theta_s, M_k).
$$

This average is the **model evidence**. A model is rewarded for fitting the data, but it is penalized if only a tiny part of its allowed parameter range fits. With equal prior model probabilities, Bayes' rule gives

$$
P(M_k\mid D)=\frac{p(D\mid M_k)}{\sum_j p(D\mid M_j)}.
$$

The calculation is similar to asking many plausible versions of each model to explain the same experiment, then comparing their average performance.

In [ ]:
labels = ['noise', 'granulation', 'oscillation']
x = np.arange(len(labels))
width = 0.36
fig, ax = plt.subplots(figsize=(8, 4))
for offset, (truth, result) in zip([-width / 2, width / 2], results.items()):
    ax.bar(x + offset, [result.probabilities[label] for label in labels], width, label=f'{truth} injection')
ax.set_xticks(x, labels)
ax.set(ylabel='Posterior model probability', ylim=(0, 1.05), title='Whole-spectrum model probabilities')
ax.legend();

## 6. What counts as a detection?

The oscillation probability is the detection probability. The other two models form a **composite null hypothesis**:

$$
P(\mathrm{no\ visible\ oscillations}\mid D)
=P(\mathrm{noise}\mid D)+P(\mathrm{granulation}\mid D).
$$

The granulation model does not assert that a solar-like star physically has no oscillations. It says that the measured PSD does not need a visible oscillation envelope once its granulation background is included. The next notebook shows why removing this model creates false detections.